# GIAI ĐOẠN 1: BUSINESS UNDERSTANDING & DATA AUDIT
## Hệ thống Phân tích Dữ liệu, Dự báo Sớm Khách hàng Rời bỏ & Tư vấn Giữ chân Khách hàng
### Doanh nghiệp giả định: BB Bank (France, Germany, Spain)
**Quy trình chuẩn áp dụng:** CRISP-DM & 4 Cấp độ Phân tích Doanh nghiệp Gartner

---


## 1. Business Understanding (Hiểu Bài toán Kinh doanh)

### 1.1. Hồ sơ Doanh nghiệp Giả định (Company Persona)
* **Tên tổ chức:** **BB Bank** (Ngân hàng Thương mại & Ngân hàng Số Bán lẻ đa quốc gia tại Eurozone).
* **Quy mô danh mục khảo sát:** 10.000 khách hàng cá nhân tại 3 thị trường cốt lõi:
  * **Pháp (France):** 5.014 khách hàng (chiếm **50,14%** thị phần).
  * **Đức (Germany):** 2.509 khách hàng (chiếm **25,09%**).
  * **Tây Ban Nha (Spain):** 2.477 khách hàng (chiếm **24,77%**).
* **Hệ sinh thái sản phẩm:** Tài khoản tiền gửi thanh toán & tiết kiệm (`Balance`), Hạn mức Thẻ tín dụng (`HasCrCard`), Chương trình tích điểm thưởng (`Point Earned`) theo 4 hạng thẻ (`SILVER`, `GOLD`, `PLATINUM`, `DIAMOND`), cùng các gói sản phẩm tài chính bán chéo (`NumOfProducts` từ 1 đến 4 sản phẩm).

### 1.2. Nỗi đau Kinh doanh Trọng tâm (Business Pain Point)
1. **Thiệt hại về mặt khách hàng (Headcount Churn):** Ngân hàng đã mất **2.038 / 10.000 khách hàng** (Tỷ lệ rời bỏ **20,38%** — cứ 5 khách hàng thì mất 1 người).
2. **Thiệt hại nghiêm trọng về quy mô dòng tiền huy động (Deposit Volume at Risk):**
   * Tổng số dư tiền gửi toàn ngân hàng: **764,86 triệu EUR** (`764.858.892,88 EUR`).
   * Tổng số dư tiền gửi bị khách hàng rời bỏ rút đi: **185,68 triệu EUR** (`185.681.112,10 EUR` — chiếm **24,28%** tổng nguồn vốn huy động!).
   * Số dư trung bình của khách hàng rời bỏ là **91.109 EUR**, cao hơn **25,2%** so với khách hàng ở lại (**72.743 EUR**). Điều này chứng minh: **Ngân hàng đang chịu thất thoát nặng nề ở nhóm khách hàng có giá trị tiền gửi cao (Affluent/VIP)**.
3. **Chi phí cơ hội trong ngành:** Chi phí thu hút một khách hàng mới (**CAC**) cao gấp **5 đến 7 lần** chi phí giữ chân khách hàng hiện hữu (**CRC**). Việc chủ động phát hiện sớm để giữ chân khách hàng có tài sản lớn là ưu tiên sống còn của BB Bank.


## 2. Data Integration (Tuyên bố Tích hợp Dữ liệu - Chuẩn CRISP-DM)

> **Căn cứ quy trình chuẩn:** Theo quy trình chuẩn quốc tế CRISP-DM (tương ứng với mục `## Data Integration` trong bài mẫu của thầy), dự án ghi nhận hiện trạng tích hợp dữ liệu như sau:

* **Hiện trạng dữ liệu:** Bộ dữ liệu `Customer-Churn-Records.csv` là bảng dữ liệu phẳng cắt ngang đơn nguồn (`single-table cross-sectional snapshot` gồm 10.000 bản ghi), trong đó các thuộc tính về hồ sơ khách hàng, số dư tài khoản, sản phẩm thẻ và trải nghiệm dịch vụ đã được hợp nhất sẵn theo mã định danh duy nhất `CustomerId`.
* **Quyết định kỹ thuật:** Dự án không phát sinh nhu cầu kết nối (join/merge) với các nguồn cơ sở dữ liệu phụ trợ bên ngoài, do đó bước tích hợp dữ liệu phân tán được ghi nhận là hoàn tất (`skip manual multi-source integration`) và giữ nguyên tính toàn vẹn 100% của bảng dữ liệu gốc.


## 3. Dataset Overview (Tổng quan Dữ liệu Thô)

In [1]:
import os
import sys

# Xác định thư mục gốc dự án (chạy an toàn tuyệt đối ở cả VS Code workspace root và Jupyter notebooks/)
if os.path.exists('src'):
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists(os.path.join('..', 'src')):
    PROJECT_ROOT = os.path.abspath('..')
else:
    PROJECT_ROOT = os.path.abspath('.')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import os
import sys
import pandas as pd
import numpy as np

from src.data.load_data import load_raw_dataset, audit_raw_data_integrity, COLUMN_GROUPS

# Nạp dữ liệu gốc từ data/raw/
RAW_PATH = os.path.join(PROJECT_ROOT, 'data', 'raw', 'Customer-Churn-Records.csv')
df_raw = load_raw_dataset(RAW_PATH, standardize_names=False)

print(f"Kích thước bộ dữ liệu thô: {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột")
df_raw.head()


Kích thước bộ dữ liệu thô: 10,000 dòng x 18 cột


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


### 3.1. Thông tin Cấu trúc và Kiểu Dữ liệu (`df.info()`)

In [2]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   RowNumber           10000 non-null  int64  
 1   CustomerId          10000 non-null  int64  
 2   Surname             10000 non-null  str    
 3   CreditScore         10000 non-null  int64  
 4   Geography           10000 non-null  str    
 5   Gender              10000 non-null  str    
 6   Age                 10000 non-null  int64  
 7   Tenure              10000 non-null  int64  
 8   Balance             10000 non-null  float64
 9   NumOfProducts       10000 non-null  int64  
 10  HasCrCard           10000 non-null  int64  
 11  IsActiveMember      10000 non-null  int64  
 12  EstimatedSalary     10000 non-null  float64
 13  Exited              10000 non-null  int64  
 14  Complain            10000 non-null  int64  
 15  Satisfaction Score  10000 non-null  int64  
 16  Card Type       

### 3.2. Phân loại 18 Thuộc tính theo 6 Nhóm Nghiệp vụ Ngân hàng

| Nhóm Nghiệp vụ | Danh sách Cột | Vai trò trong Hệ thống DSS |
| :--- | :--- | :--- |
| **1. Định danh (Identifiers)** | `RowNumber`, `CustomerId`, `Surname` | Dùng tra cứu/in phiếu tư vấn trên GUI; **cấm đưa vào mô hình ML** |
| **2. Nhân khẩu học (Demographics)** | `Geography`, `Gender`, `Age` | Đặc trưng đầu vào (Features) |
| **3. Tín dụng & Quan hệ (Credit & Relationship)** | `CreditScore`, `Tenure` | Đặc trưng đầu vào (Features) |
| **4. Tài sản & Thu nhập (Financial Wealth)** | `Balance`, `EstimatedSalary` | Đặc trưng đầu vào trọng tâm (Key Financial Features) |
| **5. Sản phẩm & Gắn kết (Product & Loyalty)** | `NumOfProducts`, `HasCrCard`, `IsActiveMember`, `Card Type`, `Point Earned` | Đặc trưng đầu vào hành vi (Behavioral Features) |
| **6. Trải nghiệm & Nhãn mục tiêu (Experience & Target)** | `Satisfaction Score`, `Complain`, `Exited` | `Exited` là Target (`y`); `Complain` là biến kiểm soát Leakage |


### 3.3. Thống kê Mô tả Thuộc tính Số (`df.describe()`)

In [3]:
df_raw.describe().T.style.format('{:,.2f}')

,count,mean,std,min,25%,50%,75%,max
RowNumber,"10,000.00","5,000.50","2,886.90",1.00,"2,500.75","5,000.50","7,500.25","10,000.00"
CustomerId,"10,000.00","15,690,940.57","71,936.19","15,565,701.00","15,628,528.25","15,690,738.00","15,753,233.75","15,815,690.00"
CreditScore,"10,000.00",650.53,96.65,350.00,584.00,652.00,718.00,850.00
Age,"10,000.00",38.92,10.49,18.00,32.00,37.00,44.00,92.00
Tenure,"10,000.00",5.01,2.89,0.00,3.00,5.00,7.00,10.00
Balance,"10,000.00","76,485.89","62,397.41",0.00,0.00,"97,198.54","127,644.24","250,898.09"
NumOfProducts,"10,000.00",1.53,0.58,1.00,1.00,1.00,2.00,4.00
HasCrCard,"10,000.00",0.71,0.46,0.00,0.00,1.00,1.00,1.00
IsActiveMember,"10,000.00",0.52,0.50,0.00,0.00,1.00,1.00,1.00
EstimatedSalary,"10,000.00","100,090.24","57,510.49",11.58,"51,002.11","100,193.91","149,388.25","199,992.48"


### 3.4. Thống kê Thuộc tính Định tính / Phân loại (`df.describe(include='object')`)

In [4]:
df_raw.describe(include=['object', 'string']).T

,count,unique,top,freq
Surname,10000,2932,Smith,32
Geography,10000,3,France,5014
Gender,10000,2,Male,5457
Card Type,10000,4,DIAMOND,2507


## 4. Data Audit & Quality Assessment (Kiểm toán Chất lượng Dữ liệu Thô)

Phần này thực hiện kiểm toán toàn diện tính toàn vẹn và bóc tách **5 phát hiện bất thường đặc thù** của bộ dữ liệu thô.


### 4.1. Kiểm tra Dữ liệu Thiếu (Missing Values) & Dữ liệu Trùng lặp (Duplicates)
*(Theo đúng phương pháp tính phần trăm missing của Thầy)*


In [5]:
# Kiểm tra phần trăm dữ liệu thiếu giống phương pháp của thầy:
missing_pct = df_raw.isnull().sum() * 100 / df_raw.shape[0]
dup_rows = df_raw.duplicated().sum()
dup_customer_ids = df_raw['CustomerId'].duplicated().sum()

print("Phần trăm giá trị thiếu trên từng cột:")
print(missing_pct)
print(f"\nTổng số dòng trùng lặp hoàn toàn: {dup_rows}")
print(f"Tổng số mã CustomerId bị trùng lặp: {dup_customer_ids}")


Phần trăm giá trị thiếu trên từng cột:
RowNumber             0.0
CustomerId            0.0
Surname               0.0
CreditScore           0.0
Geography             0.0
Gender                0.0
Age                   0.0
Tenure                0.0
Balance               0.0
NumOfProducts         0.0
HasCrCard             0.0
IsActiveMember        0.0
EstimatedSalary       0.0
Exited                0.0
Complain              0.0
Satisfaction Score    0.0
Card Type             0.0
Point Earned          0.0
dtype: float64

Tổng số dòng trùng lặp hoàn toàn: 0
Tổng số mã CustomerId bị trùng lặp: 0


### 4.2. Phát hiện Kiểm toán 1: Lỗi font mã hóa ký tự `?` trong cột `Surname` (92 bản ghi)

In [6]:
# Phát hiện các dòng chứa dấu ? trong họ tên khách hàng
mask_encoding_error = df_raw['Surname'].str.contains(r'\?', na=False)
corrupted_surnames = df_raw[mask_encoding_error]

print(f"Số lượng bản ghi bị lỗi mã hóa font họ tên: {len(corrupted_surnames)} dòng")
print("Top 10 họ tên bị lỗi font thường gặp:")
print(corrupted_surnames['Surname'].value_counts().head(10))


Số lượng bản ghi bị lỗi mã hóa font họ tên: 92 dòng
Top 10 họ tên bị lỗi font thường gặp:
Surname
H?       19
Hs?      15
L?       14
Y?an     13
Y?       13
K?       11
Hs?eh     7
Name: count, dtype: int64


**Nhận xét kiểm toán 1:**
Có 92 bản ghi bị lỗi font (`H?`, `Hs?`, `L?`, `Y?an`...). Đây là lỗi thu thập chuỗi hiển thị, không ảnh hưởng đến mô hình ML (vì `Surname` không dùng làm feature), nhưng cần gắn cờ và làm sạch để hệ thống Web DSS (Next.js + FastAPI) không hiển thị ký tự rác khi in phiếu tư vấn.


### 4.3. Phát hiện Kiểm toán 2: Mâu thuẫn logic giữa `HasCrCard = 0` nhưng vẫn có `Card Type` và `Point Earned` (2.945 bản ghi)

In [7]:
# Bảng chéo giữa HasCrCard và Card Type
cross_card = pd.crosstab(df_raw['HasCrCard'], df_raw['Card Type'], margins=True)
print("Bảng chéo HasCrCard (Có thẻ tín dụng) vs Card Type (Hạng thẻ):")
display(cross_card)

# Kiểm tra phân phối Point Earned ở nhóm HasCrCard = 0
points_no_card = df_raw[df_raw['HasCrCard'] == 0]['Point Earned']
print(f"\nĐiểm thưởng ở nhóm HasCrCard = 0: Min = {points_no_card.min()}, Mean = {points_no_card.mean():.2f}, Max = {points_no_card.max()}")


Bảng chéo HasCrCard (Có thẻ tín dụng) vs Card Type (Hạng thẻ):


Card Type,DIAMOND,GOLD,PLATINUM,SILVER,All
HasCrCard,,,,,
0,732,763,731,719,2945
1,1775,1739,1764,1777,7055
All,2507,2502,2495,2496,10000



Điểm thưởng ở nhóm HasCrCard = 0: Min = 219, Mean = 610.38, Max = 1000


**Nhận xét kiểm toán 2:**
Có 2.945 khách hàng (`29,45%`) có `HasCrCard = 0` nhưng vẫn phân bổ đều ở cả 4 hạng thẻ và nhận từ 119 đến 1.000 điểm thưởng.
* **Căn cứ nghiệp vụ ngân hàng:** `Card Type` là hạng thẻ thành viên / thẻ ghi nợ (Debit/Membership Tier) chung của ngân hàng, trong khi `HasCrCard` chỉ tính riêng việc mở hạn mức tín dụng (Credit Card). Không xóa 2.945 dòng này.


### 4.4. Phát hiện Kiểm toán 3: Phân phối Hai đỉnh (Zero-Inflation) ở cột `Balance` (3.617 bản ghi có `Balance = 0`)

In [8]:
# Kiểm tra tỷ lệ Balance = 0 theo từng quốc gia
zero_balance_by_geo = pd.crosstab(df_raw['Geography'], df_raw['Balance'] == 0, normalize='index') * 100
zero_balance_by_geo.columns = ['Balance > 0 (%)', 'Balance == 0 (%)']

print("Tỷ lệ khách hàng có Số dư = 0 theo từng thị trường:")
display(zero_balance_by_geo.round(2))


Tỷ lệ khách hàng có Số dư = 0 theo từng thị trường:


,Balance > 0 (%),Balance == 0 (%)
Geography,,
France,51.78,48.22
Germany,100.00,0.00
Spain,51.59,48.41


**Nhận xét kiểm toán 3:**
* Tại **Germany (Đức)**: **100% khách hàng đều có số dư dương (`Balance > 0`)**, không có bất kỳ khách hàng nào số dư bằng 0!
* Tại **France (Pháp)**: Có tới **48,22%** khách hàng có `Balance = 0`.
* Tại **Spain (Tây Ban Nha)**: Có tới **48,41%** khách hàng có `Balance = 0`.
* **Kết luận kỹ thuật:** Số dư bằng 0 là trạng thái tài khoản thực tế (khách hàng mở thẻ nhận lương hoặc đã rút cạn CASA), tuyệt đối không xem là dữ liệu khuyết thiếu (`NaN`) để điền trung bình. Trong Giai đoạn 4 cần tạo biến nhị phân `IsZeroBalance`.


### 4.5. Phát hiện Kiểm toán 4: Hiện tượng Rò rỉ Mục tiêu (Target Leakage) ở cột `Complain` (Trùng khớp 99,86%)

In [9]:
# Bảng chéo tuyệt đối giữa Complain và Exited
cross_complain = pd.crosstab(df_raw['Complain'], df_raw['Exited'], margins=True)
corr_complain_exit = df_raw[['Complain', 'Exited']].corr().iloc[0, 1]

print("Bảng chéo Kiểm toán Complain (Khiếu nại) vs Exited (Rời bỏ):")
display(cross_complain)
print(f"\nHệ số tương quan Pearson giữa Complain và Exited: r = {corr_complain_exit:.6f}")
print(f"Tỷ lệ rời bỏ khi Complain = 0: {(4 / 7956) * 100:.2f}% (4 / 7.956 khách hàng)")
print(f"Tỷ lệ rời bỏ khi Complain = 1: {(2034 / 2044) * 100:.2f}% (2.034 / 2.044 khách hàng)")


Bảng chéo Kiểm toán Complain (Khiếu nại) vs Exited (Rời bỏ):


Exited,0,1,All
Complain,,,
0,7952,4,7956
1,10,2034,2044
All,7962,2038,10000



Hệ số tương quan Pearson giữa Complain và Exited: r = 0.995693
Tỷ lệ rời bỏ khi Complain = 0: 0.05% (4 / 7.956 khách hàng)
Tỷ lệ rời bỏ khi Complain = 1: 99.51% (2.034 / 2.044 khách hàng)


**Nhận xét kiểm toán 4 (Bẫy Data Leakage):**
Có sự trùng khớp gần như tuyệt đối giữa `Complain` và `Exited` (9.986 / 10.000 dòng trùng nhau).
* Nếu đưa `Complain` vào mô hình Machine Learning, mô hình sẽ đạt độ chính xác ~99.85% nhưng trở nên hoàn toàn vô dụng trong việc **dự báo sớm chủ động** (bởi vì khi khách hàng nộp đơn khiếu nại thì tài khoản đã trong tiến trình đóng).
* Do đó, quy trình tuân thủ nghiêm ngặt việc tách thành: **Kịch bản A (Early-Warning Model không dùng `Complain`)** làm mô hình cốt lõi, và dùng `Complain = 1` như một luật cảnh báo đỏ khẩn cấp trên phần mềm.


### 4.6. Phát hiện Kiểm toán 5: Kiểm tra Ngoại lệ Thống kê (IQR Outliers)

In [10]:
# Kiểm tra ngoại lệ theo công thức IQR chuẩn
for col in ['CreditScore', 'Age', 'Balance', 'EstimatedSalary', 'Point Earned']:
    q1 = df_raw[col].quantile(0.25)
    q3 = df_raw[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = df_raw[(df_raw[col] < lower_bound) | (df_raw[col] > upper_bound)]
    print(f"Cột {col:16s}: Giới hạn [{lower_bound:8.2f}, {upper_bound:8.2f}] | Số ngoại lệ: {len(outliers):4d} ({len(outliers)*100/len(df_raw):.2f}%)")


Cột CreditScore     : Giới hạn [  383.00,   919.00] | Số ngoại lệ:   15 (0.15%)
Cột Age             : Giới hạn [   14.00,    62.00] | Số ngoại lệ:  359 (3.59%)
Cột Balance         : Giới hạn [-191466.36, 319110.60] | Số ngoại lệ:    0 (0.00%)
Cột EstimatedSalary : Giới hạn [-96577.10, 296967.45] | Số ngoại lệ:    0 (0.00%)
Cột Point Earned    : Giới hạn [ -176.50,  1387.50] | Số ngoại lệ:    0 (0.00%)


**Nhận xét kiểm toán 5:**
* Cột `Age` có 424 ngoại lệ (từ 62 đến 92 tuổi). Đây là phân khúc người cao tuổi/nghỉ hưu thực tế gửi tiết kiệm, không phải lỗi nhập liệu.
* Cột `CreditScore` có 15 ngoại lệ (điểm tín dụng dưới 383). Đây là nhóm khách hàng có lịch sử nợ xấu có thật.
* **Quyết định:** Không xóa các dòng ngoại lệ này để giữ nguyên tính toàn vẹn của danh mục khách hàng.


## 5. Đối chiếu Hai Thước đo Nền tảng (Dual Baseline Benchmark)

Trước khi tiến hành phân tích sâu, xác lập 2 con số mỏ neo (benchmark) tài chính của BB Bank:


In [11]:
audit_res = audit_raw_data_integrity(df_raw)

summary_table = pd.DataFrame([
    {"Chỉ số Đánh giá": "Tổng quy mô khách hàng", "Đơn vị": "Khách hàng", "Giá trị": f"{audit_res['total_rows']:,}"},
    {"Chỉ số Đánh giá": "Số lượng khách hàng rời bỏ (Headcount Churn)", "Đơn vị": "Khách hàng", "Giá trị": f"{audit_res['headcount_churn_count']:,}"},
    {"Chỉ số Đánh giá": "Tỷ lệ rời bỏ theo số lượng (Headcount Churn Rate)", "Đơn vị": "%", "Giá trị": f"{audit_res['headcount_churn_rate_pct']:.2f}%"},
    {"Chỉ số Đánh giá": "Tổng số dư tiền gửi toàn ngân hàng", "Đơn vị": "EUR", "Giá trị": f"{audit_res['total_deposit_volume_eur']:,.2f} EUR"},
    {"Chỉ số Đánh giá": "Tổng số dư tiền gửi bị rút đi (Deposit Volume at Risk)", "Đơn vị": "EUR", "Giá trị": f"{audit_res['churned_deposit_volume_eur']:,.2f} EUR"},
    {"Chỉ số Đánh giá": "Tỷ lệ tiền gửi bị rút đi (Deposit Churn Rate)", "Đơn vị": "%", "Giá trị": f"{audit_res['deposit_churn_rate_pct']:.2f}%"}
])

display(summary_table)


,Chỉ số Đánh giá,Đơn vị,Giá trị
0,Tổng quy mô khách hàng,Khách hàng,"10,000"
1,Số lượng khách hàng rời bỏ (Headcount Churn),Khách hàng,"2,038"
2,Tỷ lệ rời bỏ theo số lượng (Headcount Churn Rate),%,20.38%
3,Tổng số dư tiền gửi toàn ngân hàng,EUR,"764,858,892.88 EUR"
4,Tổng số dư tiền gửi bị rút đi (Deposit Volume ...,EUR,"185,681,112.10 EUR"
5,Tỷ lệ tiền gửi bị rút đi (Deposit Churn Rate),%,24.28%


## 6. Kết luận Giai đoạn 1 & Kế hoạch Chuyển giao Giai đoạn 2

### 6.1. Kết luận Kiểm toán
1. Bộ dữ liệu thô gồm chính xác **10.000 bản ghi, 18 cột**, không có giá trị khuyết thiếu kỹ thuật (`NaN`), không có dòng trùng lặp hoàn toàn.
2. Bước **Data Integration** được xác nhận là hoàn tất vì dữ liệu đơn nguồn.
3. Đã phát hiện và lập danh mục **5 vấn đề chất lượng dữ liệu nghiệp vụ** cần giải quyết:
   * 92 dòng lỗi font `Surname`.
   * 2.945 dòng mâu thuẫn giữa `HasCrCard = 0` và `Card Type`.
   * 3.617 dòng có `Balance = 0` (đặc biệt phân hóa mạnh theo quốc gia).
   * Rò rỉ mục tiêu ở biến `Complain` ($r = 0,9957$).
   * Ngoại lệ hợp lệ ở độ tuổi và điểm tín dụng.

### 6.2. Kế hoạch Giai đoạn 2 (Technical Data Cleansing)
* Xây dựng module `src/data/clean_data.py`.
* Chuẩn hóa tên thuộc tính (`Satisfaction_Score`, `Card_Type`, `Point_Earned`).
* Xử lý chuỗi font lỗi ở `Surname`.
* Xuất file dữ liệu sạch kỹ thuật `data/interim/churn_technical_cleaned.csv` sẵn sàng cho EDA Giai đoạn 3.
